# Exercise Parser for Chapter 7

This notebook contains functions to parse a text file of chapter exercises into a structured JSON format.

In [4]:
import re
import json


def split_questions(text):
    # Clean up spacing but DO NOT inject newlines
    text = re.sub(r"\r", "", text)

    # Ensure consistent line breaks
    text = re.sub(r"\n+", "\n", text)

    # Match only REAL question starts
    pattern = r"(?m)^\s*(\d+)\.\s+(?=[A-Z])"

    parts = re.split(pattern, text)

    questions = []

    for i in range(1, len(parts), 2):
        q_id = int(parts[i])
        q_text = parts[i + 1].strip()
        questions.append((q_id, q_text))

    return questions


def extract_subparts(q_text):
    subparts = []
    matches = list(re.finditer(r"\(([a-z])\)", q_text))

    if not matches:
        return q_text.strip(), None

    parts = re.split(r"\(([a-z])\)", q_text)

    main_question = parts[0].strip()

    for i in range(1, len(parts), 2):
        label = parts[i]
        text = parts[i + 1].strip()
        subparts.append({
            "label": label,
            "text": text
        })

    return main_question, subparts


def parse_exercises(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    split_qs = split_questions(text)

    exercises = []

    for q_id, q_text in split_qs:
        main_q, subparts = extract_subparts(q_text)

        exercise = {
            "id": q_id,
            "question": main_q
        }

        if subparts:
            exercise["subparts"] = subparts

        exercises.append(exercise)

    return {
        "chapter": "Chapter 7",
        "exercises": exercises
    }


if __name__ == "__main__":
    input_file = "ch7_end_exercises.txt"
    output_file = "exercises.json"

    data = parse_exercises(input_file)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Parsed {len(data['exercises'])} exercises → {output_file}")

Parsed 10 exercises → exercises.json


### Utility Functions

This section defines helper functions to split the raw text into individual questions and further extract any sub-parts within each question.

- `split_questions(text)`: Takes the raw text content and splits it into a list of questions, identified by a numbered pattern (e.g., `1. Question text`).
- `extract_subparts(q_text)`: Processes a single question's text to identify and extract sub-parts, typically labeled with `(a)`, `(b)`, etc.
- `parse_exercises(file_path)`: Orchestrates the parsing process by reading the file, splitting questions, and extracting sub-parts to construct a dictionary containing the chapter and a list of structured exercises.

### Output Structure (`exercises.json`)

The `exercises.json` file generated by this script will have the following structure:

```json
{
  "chapter": "Chapter 7",
  "exercises": [
    {
      "id": 1,
      "question": "Main question text",
      "subparts": [
        {
          "label": "a",
          "text": "Subpart (a) text"
        },
        {
          "label": "b",
          "text": "Subpart (b) text"
        }
      ]
    },
    // ... more exercises
  ]
}
```

Each exercise object will have an `id` and the `question` text. If an exercise contains sub-parts, a `subparts` array will be included, where each sub-part has a `label` (e.g., 'a', 'b') and its corresponding `text`.